In [1]:
import glob, os, logging
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.colors as color
import scipy.ndimage as ndimage
import matplotlib.pyplot as plt

In [2]:
# Set up the logger
# CRITICAL > ERROR > WARNING > INFO > DEBUG
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logging.basicConfig(format='%(asctime)s - %(levelname)s - %(message)s')


# User-defined functions

In [3]:
# Create directory
def create_dir(logger, dirName):
    if not os.path.exists(dirName):
        os.makedirs(dirName)
        logger.info('{} is created.'.format(dirName))
        
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

        
def process1(imgFN, outputDir, blightShown, lowThresh, highThresh, cropImg, cropPixel, showImage, saveCsv, noise=10):
    resize = 5
    
    # read rgb image
    img = Image.open(imgFN)
    
    # crop the top and bottom of image if necessary
    if cropImg == True:
        w, h = img.size
        img = img.crop((0, cropPixel, w, h-cropPixel))
    
    dpi = img.info['dpi'][0]
    r, g, b = img.split()
    
    if showImage == True:
        width, height = img.size
        img_resize = img.resize((width//resize, height//resize))
    
#     img.close()
    
    # rgb -> hsv
    r = np.array(r.convert('L'))/255.0
    g = np.array(g.convert('L'))/255.0
    b = np.array(b.convert('L'))/255.0
    rgb = np.dstack((r,g,b))
    hsv = color.rgb_to_hsv(rgb)
    h = hsv[:, :, 0] * 360
    
    # extract green pixels based on hue values
    mask = np.logical_and(h > lowThresh, h < highThresh)  # for hancock dark agriculture leaves, use 10, 200
    
    if blightShown == True:
        mask = np.logical_and(r < lowThresh, b > highThresh)  # for hancock early blight leaves, use 0.5, 0.015
    # change if scans are not working, look in other document, change h and rgb
    
    # remove single noisy pixels
    strc = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]])
    c = ndimage.binary_hit_or_miss(mask, structure1 = strc)
    c = (~c).astype(int)
    mask = mask * c
    
    # remove holes
#     mask = ndimage.binary_opening(mask, structure = np.ones((2,2))).astype(np.int)
#     mask = ndimage.binary_fill_holes(mask).astype(int) 
    
    # remove the noise chunks
    labels = ndimage.label(mask)
    lsum = ndimage.sum_labels(mask, labels[0], index=np.unique(labels[0]))
    idx = np.unique(labels[0])[(lsum > 0) & (lsum < noise)]
    mask[np.isin(labels[0], idx)] = 0
    
    # 1 square inch = 2.54 * 2.54 square centimeters
    area = (np.sum(mask) * 1.0 / dpi / dpi) * 2.54 * 2.54
    
    mask = Image.fromarray(np.uint8(mask) * 255)
    
    if saveCsv == True:
        create_dir(logger, outputDir)
        maskFN = os.path.join(outputDir, f'{os.path.basename(imgFN)[:-4]}_mask.jpg')
        mask.save(maskFN)
    
    if showImage == True:
        width, height = mask.size
        mask_resize = mask.resize((width//resize, height//resize))

        dst = Image.new('RGB', (img_resize.width + mask_resize.width, img_resize.height))
        dst.paste(img_resize, (0, 0))
        dst.paste(mask_resize, (img_resize.width, 0))
        display(dst)

    return area
    
    
def process2(inputDir, 
             outputDir, 
             blightShown = False, 
             lowThresh = 10, 
             highThresh = 200, 
             cropImg = False, 
             cropPixel = 0, 
             showImage = True, 
             saveCsv = True,
            noise = 10):
    img_list = glob.glob(inputDir + '/*[!mask].jpg')
#     img_list.extend(glob.glob('*.tif'))
#     img_list.extend(glob.glob('*.bmp'))
    
    if img_list == []:
        pass
    
    area_list = []
    sampleID_list = list()
    for imgFN in img_list:
        base_name = os.path.basename(imgFN)[:-4]
        tmp_area = process1(imgFN, outputDir, blightShown, lowThresh, highThresh, cropImg, cropPixel, showImage, saveCsv,noise)
        logger.debug(imgFN)
        logger.debug(f'{base_name}: {tmp_area} cm^2')
        
        area_list.append(tmp_area)
        sampleID_list.append(base_name)
    
    df = pd.DataFrame()
    df['filename'] = img_list
    df['sampleID'] = sampleID_list
    df['area_cm^2'] = area_list
    
    if saveCsv == True:
        fileName = os.path.join(outputDir, os.path.basename(inputDir) + '_results.csv')  # by ZWang

        df.to_csv(fileName, index = False)
        logger.info(f'File saved at {fileName}')

# Processing

In [7]:
date = 'area_processingtest_scans/PT1016_QURO.jpg'

# lowThresh = 0.565
# highThresh = 0.0115
thresh1 = 0
thresh2 = 250
noise = 1000 # pixel numbers for noise chunks

base_dir = '/Users/henryfrye/Dropbox/Intellectual_Endeavours/Wisconsin/BioSCapeTownsend/TownsendTraitFieldData/Leaf_Scans'
inputDir = os.path.join(base_dir, date)
outputDir = os.path.join(inputDir, f'{thresh1}-{thresh2}')

process2(inputDir, outputDir,
        blightShown = False, 
        lowThresh = thresh1, 
        highThresh = thresh2,
        cropImg = False, 
        cropPixel = 100, 
        showImage = False,
        saveCsv = True,
        noise = noise)

NotADirectoryError: [Errno 20] Not a directory: '/Users/henryfrye/Dropbox/Intellectual_Endeavours/Wisconsin/BioSCapeTownsend/TownsendTraitFieldData/Leaf_Scans/area_processingtest_scans/PT1016_QURO.jpg/0-250/PT1016_QURO.jpg_results.csv'